---
title: "DSPy sounds neat, but is it reliable?"
date: 2025-09-26
author: Maxime Rivest
description: "Someone asked: DSPy sounds neat, but is it reliable? Many people think they should not use DSPy if it's optimization does not work for you in practice, but that's not true. Here's why."
draft: false
format:
  html:
    toc: true
    toc-location: right
    code-tools: true
    reference-location: margin
title-block-banner: false
title-block-style: none
execute:
  echo: true  
  cache: true
  freeze: true
---

![](images/worthit.png)

Someone just asked/told me:

> DSPy sounds like a neat idea but is it practical?

TL;DR: yes.

My take on reliability and practicality is that, yes, it’s reliable—but that’s the wrong question to ask. The reason I reach for DSPy is first and foremost because it’s the simplest and most ergonomic way I know to call an LLM from Python. It’s the most lightweight way to send a prompt to an LLM, and it also happens to provide the richest tooling and possibilities: optimization of instructions and few-shots, agents, fine-tuning, structured output with validation and retries, and signatures.

Say you just want to send a string to an LLM and be productive. Without much setup or framework, you can just do this:

In [ ]:
import dspy
lm = dspy.LM("gemini/gemini-2.5-flash") # api_key="my_api_key"
lm("Count all the R's in strawberry")

['There are **3** R\'s in "strawberry".']

Nothing more, this is extremely reliable and much more ergonomic and productive than any other AI SDK I have tried (special mention to Claudette, which is also very nice).

But then you want to go from prompt to workflow or AI system. You just move to:

In [2]:
dspy.configure(lm=lm)
prg = dspy.Predict("text_input, letter -> letter_occurence: int")
prg(text_input="snowboarding is cool", letter="o")

Prediction(
    letter_occurence=3
)

That, in my experience, is as good as writing the prompt myself, but already much more general, and much more productive and ergonomic!
But then, say you want to optimize it. You add this:

In [ ]:
examples = [dspy.Example(
    text_input="snowboarding is cool", letter="o",
    letter_occurence=4
    ),
dspy.Example(
    text_input="writing the prompt", letter="i",
    letter_occurence=2
    ),
dspy.Example(
    text_input="that is extremely reliable", letter="e",
    letter_occurence=5
    )]

# mark input fields
trainset = [i.with_inputs("text_input", "letter") for i in examples]

def is_equal(gold, pred, _=None):
    return gold.letter_occurence == pred.letter_occurence

optimizer = dspy.MIPROv2(metric=is_equal)
prg_opt = optimizer.compile(prg, trainset=trainset)

Now we can use the optimized program:

In [5]:
prg_opt(text_input="Returning best identified program", letter="e")

Prediction(
    letter_occurence=4
)

That is what DSPy sent to the LLM.  For us:

The final instruction was:

And the selected few-shot example was this one:

As you can see, dspy can be used gradually and is a delight at each step of the way :)